<a href="https://colab.research.google.com/github/Terry4715/MVO-backtest/blob/main/toolKit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import os
import sys
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive/')

# Define the path to directory
data_path = '/content/drive/MyDrive/ColabNotebooks/EDHEC_Learning'

# Add the directory to sys.path to allow Python to find custom module
if data_path not in sys.path:
    sys.path.append(data_path)

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [ ]:
import pandas as pd
import os

def get_returns():
  '''
  Load the montly dataset for the top and bottle decile returns by market cap
  '''

  # Define the base path for Google Drive
  gdrive_base_path = '/content/drive/MyDrive/'
  # Construct the full path to the data file
  file_path = os.path.join(gdrive_base_path, 'ColabNotebooks', 'EDHEC_Learning', 'data', 'Portfolios_Formed_on_ME_monthly_EW.csv')

  data = pd.read_csv(file_path,
                    header=0, index_col=0, na_values=-99.99)
  rets = data[['Lo 10','Hi 10']]
  rets.index = pd.to_datetime(rets.index, format='%Y%m').to_period('M')
  rets.columns = ['SmallCap','LargeCap']
  rets = rets/100
  return rets

def drawdown(return_series: pd.Series):
  '''
  Takes a time series of asset returns.
  Computes and returns a DataFrame that contains:
  the wealth index
  the previous peaks
  percent drawdowns
  '''
  wealth_index = 1000*(1+return_series).cumprod()
  previous_peaks = wealth_index.cummax()
  drawdowns = (wealth_index - previous_peaks)/previous_peaks
  return pd.DataFrame({"Wealth": wealth_index,
                       "Previous Peak": previous_peaks,
                       "Drawdown": drawdowns})

To make `get_returns` and `drawdown` importable as a module, we need to save them into a `.py` file. The `get_returns` function within this module will be modified to construct the absolute path to your data file directly, making it self-contained.

In [ ]:
module_content = """
import pandas as pd
import os

def get_returns():
  '''
  Load the montly dataset for the top and bottle decile returns by market cap
  '''
  # Define the base path for Google Drive
  gdrive_base_path = '/content/drive/MyDrive/'
  # Construct the full path to the data file
  file_path = os.path.join(gdrive_base_path, 'ColabNotebooks', 'EDHEC_Learning', 'data', 'Portfolios_Formed_on_ME_monthly_EW.csv')

  data = pd.read_csv(file_path,
                    header=0, index_col=0, na_values=-99.99)
  rets = data[['Lo 10','Hi 10']]
  rets.index = pd.to_datetime(rets.index, format='%Y%m').to_period('M')
  rets.columns = ['SmallCap','LargeCap']
  rets = rets/100
  return rets

def drawdown(return_series: pd.Series):
  '''
  Takes a time series of asset returns.
  Computes and returns a DataFrame that contains:
  the wealth index
  the previous peaks
  percent drawdowns
  '''
  wealth_index = 1000*(1+return_series).cumprod()
  previous_peaks = wealth_index.cummax()
  drawdowns = (wealth_index - previous_peaks)/previous_peaks
  return pd.DataFrame({"Wealth": wealth_index,
                       "Previous Peak": previous_peaks,
                       "Drawdown": drawdowns})
"""

# Define the directory where the module will be saved
module_dir = os.path.join('/content/drive/MyDrive/', 'ColabNotebooks', 'EDHEC_Learning')
module_path = os.path.join(module_dir, 'edhec_risk_kit.py')

# Ensure the directory exists
os.makedirs(module_dir, exist_ok=True)

# Write the content to the file
with open(module_path, 'w') as f:
    f.write(module_content)

print(f"Module saved to: {module_path}")

Module saved to: /content/drive/MyDrive/ColabNotebooks/EDHEC_Learning/edhec_risk_kit.py


Now, we can add the directory containing our new module to `sys.path` and import it. Then, we'll use the functions from the module to demonstrate that it works.

In [ ]:
import sys

# Add the directory containing the module to sys.path
module_dir = os.path.join('/content/drive/MyDrive/', 'ColabNotebooks', 'EDHEC_Learning')
if module_dir not in sys.path:
    sys.path.append(module_dir)

# Import the module
import edhec_risk_kit as erk

# Now you can use the functions from the imported module
rets_from_module = erk.get_returns()
dd_from_module = erk.drawdown(rets_from_module['SmallCap'])

print("Returns from module (head):")
display(rets_from_module.head())

print("Drawdown from module (head, SmallCap):")
display(dd_from_module.head())

Returns from module (head):


,SmallCap,LargeCap
1926-07,-0.0145,0.0329
1926-08,0.0512,0.0370
1926-09,0.0093,0.0067
1926-10,-0.0484,-0.0243
1926-11,-0.0078,0.0270


Drawdown from module (head, SmallCap):


,Wealth,Previous Peak,Drawdown
1926-07,985.500000,985.500000,0.000000
1926-08,1035.957600,1035.957600,0.000000
1926-09,1045.592006,1045.592006,0.000000
1926-10,994.985353,1045.592006,-0.048400
1926-11,987.224467,1045.592006,-0.055822
